***Some parts of the notebook are almost the copy of [ mmta-team course](https://github.com/mmta-team/mmta_fall_2020). Special thanks to mmta-team for making them publicly available. [Original notebook](https://github.com/mmta-team/mmta_fall_2020/blob/master/tasks/01_word_embeddings/task_word_embeddings.ipynb).***

## The task of finding sentences with similar meaning

We will rank questions on [StackOverflow](https://stackoverflow.com) based on a semantic vector representation

### Embeddings

We will use pre-trained vector representations of words from Stack Overfow posts.<br>
[A word2vec model trained on Stack Overflow posts](https://github.com/vefstathiou/SO_word2vec)

In [ ]:
!wget https://zenodo.org/record/1199620/files/SO_vectors_200.bin?download=1

In [ ]:
!pip install gensim

In [ ]:
from gensim.models.keyedvectors import KeyedVectors
wv_embeddings = KeyedVectors.load_word2vec_format("SO_vectors_200.bin?download=1", binary=True)

Let's look at an example using a single word to see what an embedding is:

In [ ]:
word = 'dog'
if word in wv_embeddings:
    print(wv_embeddings[word].dtype, wv_embeddings[word].shape)

float32 (200,)


In [ ]:
print(f"Num of words: {len(wv_embeddings.index_to_key)}")

Num of words: 1787145


Let's find the words most similar to the word `dog` and check is the word "cats" among it

In [ ]:
word = "cats"
topn=5
dog_most_sim = wv_embeddings.most_similar('dog', topn=topn)
print(dog_most_sim)

for i in range(len(dog_most_sim)):
    if dog_most_sim[i][0] == word:
      print(f'yes, word "{word}" is in top {topn} similar words to the word "dog", ', 'place:', i)

[('animal', 0.8564180135726929), ('dogs', 0.7880866527557373), ('mammal', 0.7623804211616516), ('cats', 0.7621253728866577), ('animals', 0.760793924331665)]
yes, word "cats" is in top 5 similar words to the word "dog",  place: 3


### Vector representation of text

Let’s move from vector representations of individual words to vector representations of questions, defined as the **average** of the vectors of all words in the question. If there is no pre-trained vector for a particular word, it should be skipped. If a question does not contain any known words, a zero vector should be returned.

In [ ]:
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

class MyTokenizer:
    def __init__(self):
        pass
    def tokenize(self, text):
      return [w.lower() for w in word_tokenize(text) if w.isalnum()]

tokenizer = MyTokenizer()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
def question_to_vec(question, embeddings, tokenizer, dim=200):
    """
        question: string
        embeddings: our vector representation
        dim: the size of every vector in our representation

        return: vector representation of a question
    """
    feature = np.zeros(dim, dtype='float32')
    cnt = 0
    question_tokens = tokenizer.tokenize(question)

    for token in question_tokens:
      if token in embeddings:
        feature += embeddings[token]
        cnt += 1
    if cnt != 0:
      question_vec = feature / cnt
    else:
      question_vec = feature
    return question_vec

Now we can create a vector representation of any sentence

Let's determine which component is the third (index 2) of the vector representing the sentence `I love neural networks`.

In [ ]:
question = "I love neural networks"
question_vec = question_to_vec(question, wv_embeddings, tokenizer)
print(question_vec[2])

-1.2854122


### Assessing Text Similarity

Let's assume we're using ideal word vector representations. In that case cosine distance between duplicate sentences should be smaller than that between randomly selected sentences.

For each of the $N$ questions, let’s generate $R$ random negative examples and mix in the actual duplicates as well. For each question, we’ll rank $R + 1$ examples using our model and look at the position of the duplicate. We want the duplicate to be first in the ranked list.

#### Hits@K
The first simple metric is the number of correct hits for some $K$:
$$ \text{Hits@K} = \frac{1}{N}\sum_{i=1}^N \, [rank\_q_i^{'} \le K],$$
* $\begin{equation*}
[x < 0 ] \equiv
 \begin{cases}
   1, &x < 0\\
   0, &x \geq 0
 \end{cases}
\end{equation*}$ - indicator func
* $q_i$ - the $i$-th question
* $q_i^{'}$ - its duplicate
* $rank\_q_i^{'}$ - the position of the duplicate in the ranked list of the closest suggestions for question $q_i$.

Hits@K measures the proportion of questions for which the correct answer ranked within the top K positions among the ranked candidates.

#### DCG@K
The second metric is a simplified version of the DCG metric that accounts for the order of items in the list by multiplying an item’s relevance by a weight equal to the inverse logarithm of its rank:
$$ \text{DCG@K} = \frac{1}{N} \sum_{i=1}^N\frac{1}{\log_2(1+rank\_q_i^{‘})}\cdot[rank\_q_i^{’} \le K],$$
With this metric, the model is penalized for a high rank of the correct answer.

DCG@K measures ranking quality by considering not only the fact that the correct answer is in the top-K, but also ***its exact position***.

Let's compute `DCG@10` if $rank\_q_i^{'} = 9$ (round to one decimal place):

DCG@10 will be equal to 1/(log₂(1+9)) * [9 ≤ 10]



In [ ]:
import math
print(round(1/(math.log2(1+9)),1))

0.3


Let's find out the max result of `Hits@47 - DCG@1':



To achieve the maximum result, we need the maximum Hits@47 and the minimum DCG@1. The minimum DCG@1 will be 0 if the duplicate’s rank is greater than 1. Accordingly, if the duplicate’s rank is >= 2 and less than 47, then Hits@47 will be equal to 1. Answer: 1

### HITS\_COUNT and DCG\_SCORE

Each function has two arguments: $dup\_ranks$ and $k$.

$dup\_ranks$ is a list that contains the ranks of the duplicates (their positions in the sorted list).

For example, for “What is the Python language?” $dup\_ranks = [2]$.

In [ ]:
def hits_count(dup_ranks, k):
    """
        dup_ranks: list of duplicates indicies
        k: threshold value for rank
        result: return Hits@k
    """
    hits_value = sum(1 for rank in dup_ranks if rank <= k)
    hits_value /= len(dup_ranks)
    return hits_value

In [ ]:
dup_ranks = [2]

k = 1
hits_value = hits_count(dup_ranks, k)
print(f"Hits@1 = {hits_value}")

k = 4
hits_value = hits_count(dup_ranks, k)
print(f"Hits@4 = {hits_value}")

Hits@1 = 0.0
Hits@4 = 1.0


In [ ]:
import math

def dcg_score(dup_ranks, k):
    """
        dup_ranks: list of duplicates indicies
        k: threshold value for rank
        result: return DCG@k
    """
    # Calculate the sum for all relevant duplicates
    dcg_value = 0
    for rank in dup_ranks:
      if rank <= k:
        dcg_value += 1/ (math.log2(1 + rank))

    # Divide by the total number of questions
    dcg_value /= len(dup_ranks)
    return dcg_value

In [ ]:
# Example of a list of duplicate items
dup_ranks = [2]

# Compute DCG@1
dcg_value = dcg_score(dup_ranks, k=1)
print(f"DCG@1 = {dcg_value:.3f}")

# Compute DCG@4
dcg_value = dcg_score(dup_ranks, k=4)
print(f"DCG@4 = {dcg_value:.3f}")

DCG@1 = 0.000
DCG@4 = 0.631


Let's test the functions. Let $N = 1$, meaning one experiment. We'll search for a copy of the question and evaluate the metrics

In [ ]:
import pandas as pd

In [ ]:
copy_answers = ["How does the catch keyword determine the type of exception that was thrown",]

# our candidates
candidates_ranking = [["How Can I Make These Links Rotate in PHP",
                       "How does the catch keyword determine the type of exception that was thrown",
                       "NSLog array description not memory address",
                       "PECL_HTTP not recognised php ubuntu"],]

# dup_ranks — the positions of our copies; since there is only one experiment, this array has a length of 1
dup_ranks = [2]

# compute a metric for different k
print('HIT:', [hits_count(dup_ranks, k) for k in range(1, 5)])
print('DCG:', [round(dcg_score(dup_ranks, k), 5) for k in range(1, 5)])

HIT: [0.0, 1.0, 1.0, 1.0]
DCG: [0.0, 0.63093, 0.63093, 0.63093]


### Data
[arxiv link](https://drive.google.com/file/d/1QqT4D0EoqJTy7v9VrNCYD-m964XZFR7_/edit)

`train.tsv` - training dataset.<br> Each line contains the following fields, separated by tabs: **<question>, <similar question>**

`validation.tsv` - the validation dataset.<br> Each line contains the following, separated by tabs: **<question>, <similar question>, <negative example 1>, <negative example 2>, ...**

In [ ]:
!unzip stackoverflow_similar_questions.zip

In [ ]:
def read_corpus(filename):
    data = []
    with open(filename, encoding='utf-8') as file:
        for line in file:
            data.append(line.strip().split('\t'))
    return data

In [ ]:
validation_data = read_corpus('./data/validation.tsv')

Num of lines:

In [ ]:
len(validation_data)

3760

The size of the first few lines:

In [ ]:
for i in range(25):
    print(i + 1, len(validation_data[i]))

1 1001
2 1001
3 1001
4 1001
5 1001
6 1001
7 1001
8 1001
9 1001
10 1001
11 1001
12 1001
13 1001
14 1001
15 1001
16 1001
17 1001
18 1001
19 1001
20 1001
21 1001
22 1001
23 1001
24 1001
25 1001


### Ranking Without Training

Implement a function to rank candidates based on cosine distance. Given a list of candidates, the function should return a sorted list of pairs (position in the original list of candidates, candidate). The candidate’s position in the resulting list represents its rank (first is best). For example, if the original list of candidates was [a, b, c], and the candidate most similar to the original question is c, followed by a, and finally b, then the function should return the list **[(2, c), (0, a), (1, b)]**.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from copy import deepcopy

In [ ]:
def rank_candidates(question, candidates, embeddings, tokenizer, dim=200):
    """
        question: string
        candidates: array of strings (candidates) [a, b, c]
        result: tuples (start position, candidate) [(2, c), (0, a), (1, b)]
    """
    result = []
    question_vec = question_to_vec(question, embeddings, tokenizer).reshape(1, -1)

    for i in range(len(candidates)):
      cand_vec = question_to_vec(candidates[i], embeddings, tokenizer).reshape(1, -1)
      cos_sim = cosine_similarity(question_vec, cand_vec)[0][0]
      candidate = (cos_sim, i,  candidates[i])
      result.append(candidate)
    result.sort(key = lambda x: x[0], reverse=True)
    result = [x[1:] for x in result]
    return result

Let's test how the function works using the examples below. Let $N=2$.

In [ ]:
questions = ['converting string to list', 'Sending array via Ajax fails']

candidates = [['Convert Google results object (pure js) to Python object', # first experiment
               'C# create cookie from string and send it',
               'How to use jQuery AJAX for an outside domain?'],

              ['Getting all list items of an unordered list in PHP',      # second experiment
               'WPF- How to update the changes in list item of a list',
               'select2 not displaying search results']]

In [ ]:
for question, q_candidates in zip(questions, candidates):
        ranks = rank_candidates(question, q_candidates, wv_embeddings, tokenizer)
        print(ranks)
        print()

[(1, 'C# create cookie from string and send it'), (0, 'Convert Google results object (pure js) to Python object'), (2, 'How to use jQuery AJAX for an outside domain?')]

[(0, 'Getting all list items of an unordered list in PHP'), (1, 'WPF- How to update the changes in list item of a list'), (2, 'select2 not displaying search results')]



Now we can evaluate the quality of our method. For validation, let's use 1,000 examples

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, wv_embeddings, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.394 | Hits@   1: 0.394
DCG@   5: 0.480 | Hits@   5: 0.556
DCG@  10: 0.503 | Hits@  10: 0.628
DCG@ 100: 0.547 | Hits@ 100: 0.846
DCG@ 500: 0.563 | Hits@ 500: 0.966
DCG@1000: 0.567 | Hits@1000: 1.000


From the formulas above, we can see that

- $ \text{Hits@K} $ is a **monotonically non-decreasing function** of $ K $ that tends to 1 as $ K \to \infty $.

- $ \text{DCG@K} $ is a **monotonically non-decreasing function** of $ K $, but its growth slows as $ K $ increases due to the decreasing weight $ \frac{1}{\log_2(1 + \text{rank}_{q'_i})} $.

### Embeddings trained on a corpus of similar questions

In [ ]:
train_data = read_corpus('./data/train.tsv')

Let’s improve the model’s quality.<br> To do this, we’ll concatenate the questions into pairs and train the Word2Vec model from gensim on them.

***Let’s take a closer look*** at this concatenation process.

1. Each line from `train_data` is split into a question and a list of candidates.

2. For each candidate, the question is concatenated with it into a single line.

3. The concatenated line (`combined_text`) is tokenized, and the resulting list of tokens is added to the overall corpus (`corpus`).

***Example***

    Question: “What is Python?”
    Candidates: [“Python is a programming language”, “Java is another language”]
    Combined strings:
        “What is Python? Python is a programming language”
        “What is Python? Java is another language”

    Tokenized lists:
        [‘what’, ‘is’, ‘python’, ‘python’, ‘is’, ‘a’, ‘programming’, ‘language’]
        [‘what’, ‘is’, ‘python’, ‘java’, ‘is’, ‘another’, ‘language’]



In [ ]:
train_data[111258]

['Determine if the device is a smartphone or tablet?',
 'Change imageView params in all cards together']

In [ ]:
# Creating a Common Text Corpus
corpus = []

for line in tqdm(train_data):
    question = line[0]        # Initial question
    candidates = line[1:]     # List of duplicates / similar questions

    for cand in candidates:
        # Concatenate the original question with the candidate's name
        combined_text = question + " " + cand
        # Tokenize the string we received
        tokens = tokenizer.tokenize(combined_text)
        corpus.append(tokens)



  0%|          | 0/1000000 [00:00<?, ?it/s]

In [ ]:
from gensim.models import Word2Vec
from tqdm.notebook import tqdm

embeddings_trained = Word2Vec(
    sentences=corpus,        # Corpus of tokenized texts
    vector_size=200,         # Vector dimension (matches Part 1)
    window=5,                # Context window size
    min_count=2,             # Minimum word frequency
    workers=4                # Number of threads for parallel processing
).wv

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.288 | Hits@   1: 0.288
DCG@   5: 0.369 | Hits@   5: 0.441
DCG@  10: 0.390 | Hits@  10: 0.508
DCG@ 100: 0.442 | Hits@ 100: 0.761
DCG@ 500: 0.465 | Hits@ 500: 0.942
DCG@1000: 0.471 | Hits@1000: 1.000


Let's try another approach to increase the metrics

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Collect all available question texts for TF-IDF training
all_questions = []

# Add questions from train
for line in train_data:
    all_questions.extend(line)

# Add questions from validation
for line in validation_data:
    all_questions.extend(line)

# Init and train TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(tokenizer=tokenizer.tokenize, lowercase=True)
tfidf_vectorizer.fit(all_questions)

# Create a dictionary for quick lookup: word → ID in the TF-IDF matrix
tfidf_vocab = tfidf_vectorizer.vocabulary_
# IDF values for each word
idf_weights = dict(zip(tfidf_vectorizer.get_feature_names_out(), tfidf_vectorizer.idf_))

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
def question_to_vec_tfidf(question, embeddings, tokenizer, tfidf_weights, dim=200):
    """
    Converts a question into a vector, taking into account TF-IDF word weights.

    question: a string containing the question
    embeddings: a pre-trained embedding model (SO_vectors_200.bin)
    tokenizer: an instance of the tokenizer
    tfidf_weights: a dictionary of the form {word: idf_weight}
    dim: the dimension of the embeddings
    """
    feature = np.zeros(dim, dtype='float32')
    total_weight = 0.0

    tokens = tokenizer.tokenize(question)

    for token in tokens:
        if token in embeddings:
            # Calculating the TF-IDF (IDF) weight of a word
            weight = tfidf_weights.get(token, 1.0)

            # Add the weighted vector of the word
            feature += embeddings[token] * weight
            total_weight += weight

    if total_weight > 0:
        question_vec = feature / total_weight
    else:
        question_vec = feature

    return question_vec

In [ ]:
def rank_candidates_tfidf(question, candidates, embeddings, tokenizer, tfidf_weights, dim=200):
    result = []
    question_vec = question_to_vec_tfidf(question, embeddings, tokenizer, tfidf_weights, dim).reshape(1, -1)

    for i, candidate_text in enumerate(candidates):
        cand_vec = question_to_vec_tfidf(candidate_text, embeddings, tokenizer, tfidf_weights, dim).reshape(1, -1)
        cos_sim = cosine_similarity(question_vec, cand_vec)[0][0]
        result.append((cos_sim, i, candidate_text))

    # Sort by descending cosine similarity
    result.sort(key=lambda x: x[0], reverse=True)
    return [x[1:] for x in result]

In [ ]:
wv_ranking_tfidf = []
max_validation_examples = 1000

for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates_tfidf(q, ex, wv_embeddings, tokenizer, idf_weights)
    wv_ranking_tfidf.append([r[0] for r in ranks].index(0) + 1)

  0%|          | 0/3760 [00:00<?, ?it/s]

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking_tfidf, k), k, hits_count(wv_ranking_tfidf, k)))

  0%|          | 0/6 [00:00<?, ?it/s]

DCG@   1: 0.457 | Hits@   1: 0.457
DCG@   5: 0.538 | Hits@   5: 0.611
DCG@  10: 0.560 | Hits@  10: 0.681
DCG@ 100: 0.600 | Hits@ 100: 0.878
DCG@ 500: 0.613 | Hits@ 500: 0.974
DCG@1000: 0.615 | Hits@1000: 1.000


## Conclusion:

* **Which embeddings perform better on this task, and why?**

  Typically the pretrained SO_vectors_200 model (trained on millions of StackOverflow posts) outperforms both the from-scratch Word2Vec trained only on train_data and, often, is comparable to or slightly beaten by the IDF-weighted version. The reasons:

  * The custom-trained embeddings are trained on a much smaller, narrower corpus, so rare/technical tokens get poor vector estimates.
  * IDF weighting helps by downweighting generic high-frequency words ("how", "does", "the") that contribute little to distinguishing duplicate questions, letting distinctive technical terms dominate the similarity score.

* **Why did the solution to this problem yield poor results?**

  Averaging word vectors is a fairly weak sentence representation: it discards word order, syntax, and negation, and treats "How to catch an exception" the same as "exception catch how to". It also fully drops OOV tokens (code snippets, rare API names, typos) rather than handling them, which is common in programming Q&A text. Finally, cosine similarity on averaged vectors rewards topical/lexical overlap more than genuine semantic equivalence, so many near-duplicate-but-different-topic questions rank confusingly close together.
